# A/B Testing Experiment for ML Models

This notebook adds an experimental evaluation layer to the existing ML workflow by treating:

- **Control**: Logistic Regression
- **Treatment**: Random Forest

We cover hypothesis testing, confidence intervals, power analysis, and conversion-rate A/B simulation.

## 1) Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

sns.set_theme(style="whitegrid")
np.random.seed(42)

## 2) Define Hypothesis

- **H0**: The new model does not improve prediction accuracy.
- **H1**: The new model improves prediction accuracy.

## 3) Simulate Control vs Treatment Model Scores

We simulate cross-validation accuracy samples where treatment has a small lift.

In [ ]:
n_folds = 30
control_scores = np.random.normal(loc=0.835, scale=0.018, size=n_folds)
treatment_scores = np.random.normal(loc=0.848, scale=0.018, size=n_folds)

scores_df = pd.DataFrame({
    'score': np.concatenate([control_scores, treatment_scores]),
    'group': ['Control (Logistic Regression)'] * n_folds + ['Treatment (Random Forest)'] * n_folds
})
scores_df.head()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=scores_df, x='group', y='score', palette=['#4E79A7', '#F28E2B'])
plt.title('Model Accuracy Distribution: Control vs Treatment')
plt.xlabel('')
plt.ylabel('Accuracy')
plt.xticks(rotation=8)
plt.show()

## 4) Hypothesis Testing (Two-sample t-test)

In [ ]:
t_stat, p_value_two_sided = stats.ttest_ind(treatment_scores, control_scores, equal_var=False)
p_value_one_sided = p_value_two_sided / 2 if t_stat > 0 else 1.0 - (p_value_two_sided / 2)

print(f't-statistic: {t_stat:.4f}')
print(f'p-value (two-sided): {p_value_two_sided:.6f}')
print(f'p-value (one-sided, treatment > control): {p_value_one_sided:.6f}')

## 5) Confidence Intervals (95%)

We compute 95% CI for each group mean and for the mean difference.

In [ ]:
def mean_ci(values, alpha=0.05):
    values = np.array(values)
    n = len(values)
    mean = values.mean()
    sem = stats.sem(values)
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    margin = t_crit * sem
    return mean, mean - margin, mean + margin

control_mean, control_lo, control_hi = mean_ci(control_scores)
treatment_mean, treatment_lo, treatment_hi = mean_ci(treatment_scores)
diff = treatment_scores - control_scores
diff_mean, diff_lo, diff_hi = mean_ci(diff)

print(f'Control mean accuracy: {control_mean:.4f} (95% CI: {control_lo:.4f}, {control_hi:.4f})')
print(f'Treatment mean accuracy: {treatment_mean:.4f} (95% CI: {treatment_lo:.4f}, {treatment_hi:.4f})')
print(f'Mean lift (T - C): {diff_mean:.4f} (95% CI: {diff_lo:.4f}, {diff_hi:.4f})')

In [ ]:
summary_plot = pd.DataFrame({
    'group': ['Control', 'Treatment'],
    'mean': [control_mean, treatment_mean],
    'low': [control_lo, treatment_lo],
    'high': [control_hi, treatment_hi]
})

plt.figure(figsize=(7, 5))
plt.errorbar(
    x=summary_plot['group'],
    y=summary_plot['mean'],
    yerr=[summary_plot['mean'] - summary_plot['low'], summary_plot['high'] - summary_plot['mean']],
    fmt='o', capsize=8, color='#2C7FB8'
)
plt.title('Mean Accuracy with 95% Confidence Intervals')
plt.ylabel('Accuracy')
plt.ylim(0.80, 0.88)
plt.show()

## 6) Power Analysis

Estimate required sample size for 80% power and alpha=0.05 using observed effect size.

In [ ]:
effect_size = (treatment_mean - control_mean) / np.sqrt((np.var(treatment_scores, ddof=1) + np.var(control_scores, ddof=1)) / 2)
analysis = TTestIndPower()
required_n = analysis.solve_power(effect_size=effect_size, power=0.8, alpha=0.05, ratio=1.0, alternative='two-sided')

print(f'Observed effect size (Cohen\'s d): {effect_size:.4f}')
print(f'Required sample size per group (80% power): {np.ceil(required_n):.0f}')

## 7) A/B Conversion Simulation

Simulate online experiment conversions and evaluate with z-test.

In [ ]:
n_control = 5000
n_treatment = 5000
p_control = 0.08
p_treatment = 0.10

conv_control = np.random.binomial(n=n_control, p=p_control)
conv_treatment = np.random.binomial(n=n_treatment, p=p_treatment)

counts = np.array([conv_treatment, conv_control])
nobs = np.array([n_treatment, n_control])
z_stat, p_val = proportions_ztest(count=counts, nobs=nobs, alternative='larger')

rate_control = conv_control / n_control
rate_treatment = conv_treatment / n_treatment

ci_control = proportion_confint(conv_control, n_control, alpha=0.05, method='wilson')
ci_treatment = proportion_confint(conv_treatment, n_treatment, alpha=0.05, method='wilson')

print(f'Control: {conv_control}/{n_control} = {rate_control:.4%}, 95% CI={ci_control}')
print(f'Treatment: {conv_treatment}/{n_treatment} = {rate_treatment:.4%}, 95% CI={ci_treatment}')
print(f'Lift: {(rate_treatment - rate_control):.4%}')
print(f'z-statistic: {z_stat:.4f}')
print(f'p-value (one-sided): {p_val:.6f}')

In [ ]:
n_sim = 1000
sim_lifts = []
sim_pvals = []

for _ in range(n_sim):
    c = np.random.binomial(n=n_control, p=p_control)
    t = np.random.binomial(n=n_treatment, p=p_treatment)
    lift = (t / n_treatment) - (c / n_control)
    z, p = proportions_ztest([t, c], [n_treatment, n_control], alternative='larger')
    sim_lifts.append(lift)
    sim_pvals.append(p)

significant_rate = np.mean(np.array(sim_pvals) < 0.05)

print(f'Simulated significant detection rate (empirical power): {significant_rate:.3f}')

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(sim_lifts, bins=35, kde=True, color='#59A14F')
plt.axvline(np.mean(sim_lifts), color='black', linestyle='--', label='Mean lift')
plt.title('Distribution of Simulated Conversion Lift (Treatment - Control)')
plt.xlabel('Conversion Lift')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## 8) Experimentation Framework

1. Define hypothesis
2. Define metrics
3. Run experiment
4. Evaluate statistical significance
5. Measure confidence intervals
6. Verify power and sample size assumptions